# Reproduce the publication figure set

Validates the manifest and executes the four topic notebooks in canonical order. Figure saving remains disabled unless explicitly enabled.

In [ ]:
from pathlib import Path
import json, os
ROOT=Path.cwd().resolve(); ROOT=ROOT.parent if ROOT.name=="experiments" else ROOT
RUN_PROFILE=os.getenv("MNN_RUN_PROFILE","reduced")
DEVICE=os.getenv("MNN_DEVICE","auto")
WORKERS=os.getenv("MNN_WORKERS","auto")
SAVE_FIGURES=os.getenv("MNN_SAVE_FIGURES","0")=="1"
OUTPUT_DIR=Path(os.getenv("MNN_OUTPUT_DIR",str(ROOT/"experiments"/"generated_figures")))
OVERWRITE=os.getenv("MNN_OVERWRITE","0")=="1"
RUN_EXTERNAL_DATA=os.getenv("MNN_RUN_EXTERNAL_DATA","0")=="1"
ALLOW_DATA_DOWNLOADS=os.getenv("MNN_ALLOW_DATA_DOWNLOADS","0")=="1"
assert RUN_PROFILE in {"reduced","publication","smoke"}
manifest=json.loads((ROOT/"experiments"/"figure_manifest.json").read_text())
figures=manifest["figures"]
assert manifest["figure_count"] == len(figures) == 17
assert manifest["active_manuscript_count"] == sum(x["tier"]=="main" for x in figures) == 7
assert len({x["filename"] for x in figures}) == 17
assert all(x["provenance_class"] in {"live-exact","live-reduced","published-sample-archive","published-aggregate","immutable-authored","external-gated","reference-only","placeholder"} for x in figures)

RUN_NOTEBOOKS=os.getenv("MNN_REPRODUCE_EXECUTE","1")=="1"
if RUN_NOTEBOOKS:
    import nbformat
    notebooks=["01_device_and_static.ipynb","02_homeostasis.ipynb","03_temporal_memory.ipynb","04_representations.ipynb"]
    old=os.getcwd(); os.chdir(ROOT/"experiments")
    try:
        for name in notebooks:
            print(f"Executing {name} in the current kernel (saving remains opt-in)")
            child=nbformat.read(name,as_version=4); namespace={"__name__":"__main__"}
            for index,cell in enumerate(child.cells):
                if cell.cell_type == "code":
                    exec(compile(cell.source,f"{name}:cell{index}","exec"),namespace)
            expected={x["filename"] for x in figures if x["notebook"]==name}
            emitted={x["filename"] for x in namespace["FIGURE_REPORT"]}
            assert emitted == expected, (name,sorted(expected-emitted),sorted(emitted-expected))
    finally: os.chdir(old)
print("Validated 17 supported figures (7 active manuscript figures); no figure files are required.")
